In [ ]:
import re
import json
import argparse
import os
import sys
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

SLIDES TEXT INPUT

In [ ]:
sample_id = 1

with open("./Agent_resources/Text_versions/pid{}_{}_text_1.txt".format(course_id,material_id), "r", encoding="utf-8") as f:
    slides_text = f.read()

ALL CONTENTS PROCESSING

In [ ]:
# Execute this code with all your dataset samples generated in the previous notebook EXTRACT_TEXT_AND_LABELS
# You'll need to run it with sample_id values of 1,2,3,4 by defect
sample_id = 1
# This code works for the course "Digital Signal Theory" from Kyushu University used in the experiments
# For a new course, you need to include here the information about the lecture materials included in the course (contentsid)
# the number of pages of each material (pages) and the name of the corresponding class (title)
course_id = 207
course_materials = pd.read_csv("./pid{}_material_info.csv".format(course_id))[["contentsid","pages","title"]]

for n_material in range(course_materials.shape[0]):
    material_id = course_materials["contentsid"][n_material]
    max_pages = course_materials["pages"][n_material]
    title = course_materials["title"][n_material]
    print("ID: {}".format(material_id))
    print("N pages: {}".format(max_pages))
    print("Title: {}".format(title))

    with open("./Agent_resources/Text_versions/pid{}_{}_text_{}.txt".format(course_id,material_id,sample_id), "r", encoding="utf-8") as f:
        slides_text = f.read()

    pattern = r"SLIDE \d+:\s*(.*?)(?=\n)(.*?)(?=SLIDE \d+:|$)"
    slides = re.findall(pattern, slides_text, flags=re.DOTALL)

    slide_types = []
    refined_text = ""
    with open(f"./Agent_resources/Slide_types/pid_{course_id}_{material_id}_slide_types.txt", "r") as file:
        for line in file:
            slide_types.append(line.strip())

    for i, (heading, content) in enumerate(slides, 1):
        title = heading.strip()
        body = content.strip()

        with open("./Agent_resources/TitleC_versions/pid{}_{}_title_{}_p{}.txt".format(course_id,material_id,sample_id,i), "w", encoding="utf-8") as f:
            f.write(title)
        with open("./Agent_resources/TitleC_versions/pid{}_{}_body_{}_p{}.txt".format(course_id,material_id,sample_id,i), "w", encoding="utf-8") as f:
            f.write(body)


        if(slide_types[i-1]=="TITLE_SLIDE"):
            refined_text += f"------ Slide {i} (HEADER) ------"
            refined_text += f"\n{heading.strip()}"
        elif(slide_types[i-1]=="HEADER_SLIDE"):
            refined_text += f"------ Slide {i} (HEADER) ------"
            refined_text += f"\n{heading.strip()}"
        else:
            refined_text += f"\n------ Slide {i} ------"
            refined_text += f"\n{heading.strip()}:"
            refined_text += f"\n{content.strip()}"

    with open("./Agent_resources/TextR_versions/pid{}_{}_rtext_{}.txt".format(course_id,material_id,sample_id), "w", encoding="utf-8") as f:
        f.write(refined_text)